# 阶段3验证：解释器、统一截面清洗与5日指标

本 Notebook 读取 `data/processed/` 中的六特征数据和点时申万行业长表，不会重新计算 VWAP、清洗原始行情或写入数据。请先运行 `prepare_daily_data.ipynb` 和 `prepare_industry_data.ipynb`。为控制内存，验证默认使用最近500日、最多1000只股票的已处理样本。

In [ ]:
from pathlib import Path
import json
import sys
import time
import numpy as np

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from factor_gfn.evaluator import (
    FEATURE_NAMES, INTERPRETER_OPERATOR_FUNCTIONS, EvaluationConfig,
    FactorInterpreter, build_forward_returns, clean_candidate_factor_cross_sections, evaluate_rank_ic,
    excess_return_correlation, factor_cross_sectional_correlation,
    infer_long_direction, long_portfolio_series, summarize_correlation,
    summarize_excess_returns,
)
from factor_gfn.data import load_sw_industry_panel
from factor_gfn.grammar import Expression, NON_LEAF_OPERATORS, get_action_id

processed_dir = project_root / 'data' / 'processed'
paths = {name: processed_dir / name for name in [
    'data_tensor.npy', 'valid_mask.npy', 'universe_mask.npy',
    'date_list.npy', 'stock_list.npy', 'metadata.json',
]}
industry_path = processed_dir / 'industry_sw_daily.parquet'
missing = [str(path) for path in paths.values() if not path.exists()]
assert not missing, f'请先运行 prepare_daily_data.ipynb，缺少：{missing}'
assert industry_path.exists(), f'请先运行 prepare_industry_data.ipynb：{industry_path}'

In [ ]:
data_tensor = np.load(paths['data_tensor.npy'], mmap_mode='r')
valid_mask = np.load(paths['valid_mask.npy'], mmap_mode='r')
universe_mask = np.load(paths['universe_mask.npy'], mmap_mode='r')
date_list = np.load(paths['date_list.npy'], allow_pickle=False)
stock_list = np.load(paths['stock_list.npy'], allow_pickle=False)
metadata = json.loads(paths['metadata.json'].read_text(encoding='utf-8'))
industry_labels = load_sw_industry_panel(date_list, stock_list, level=1, path=industry_path)

assert data_tensor.shape == (len(date_list), 6, len(stock_list))
assert valid_mask.shape == universe_mask.shape == (len(date_list), len(stock_list))
assert tuple(metadata['feature_order']) == FEATURE_NAMES
assert data_tensor.dtype.kind == 'f'
usable_mask = np.asarray(valid_mask & universe_mask)

day_slice = slice(max(0, len(date_list) - 500), len(date_list))
stock_slice = slice(0, min(1000, len(stock_list)))
sample_tensor = np.asarray(data_tensor[day_slice, :, stock_slice])
sample_usable = usable_mask[day_slice, stock_slice]
sample_dates = date_list[day_slice]
sample_stocks = stock_list[stock_slice]
sample_industries = industry_labels[day_slice, stock_slice]
print('完整张量:', data_tensor.shape, data_tensor.dtype)
print('验证样本:', sample_tensor.shape, sample_dates[0], '至', sample_dates[-1])
print('样本可用率:', float(sample_usable.mean()))
print('样本申万一级覆盖率:', float(np.mean(sample_industries >= 0)))

In [ ]:
# 在更小切片上确认52个算子全部可由解释器访问，避免一次产生过多中间矩阵。
operator_tensor = sample_tensor[-100:, :, :min(200, sample_tensor.shape[2])]
operator_interpreter = FactorInterpreter(operator_tensor)
assert len(INTERPRETER_OPERATOR_FUNCTIONS) == 52
for operator in NON_LEAF_OPERATORS:
    window = 5 if operator.requires_window else 0
    children = [get_action_id('close')]
    if operator.arity == 2:
        children.append(get_action_id('volume'))
    expression = Expression.from_prefix((get_action_id(operator.name, window), *children))
    output = operator_interpreter.evaluate(expression)
    assert output.shape == (operator_tensor.shape[0], operator_tensor.shape[2])
    assert not np.isinf(output).any()
print('52个算子真实数据切片检查通过')

In [ ]:
expressions = [
    Expression.from_prefix((get_action_id('cs_rank'), get_action_id('ts_delta', 5), get_action_id('close'))),
    Expression.from_prefix((get_action_id('neg'), get_action_id('ts_std', 10), get_action_id('close'))),
]
config = EvaluationConfig(min_cross_section_count=20)
forward_returns = build_forward_returns(sample_tensor[:, 0, :], config)
factor_results = []
start = time.perf_counter()
for expression in expressions:
    factor = FactorInterpreter(sample_tensor).evaluate(expression)
    factor_before = factor.copy()
    cleaned = clean_candidate_factor_cross_sections(factor, sample_industries, sample_usable)
    np.testing.assert_allclose(factor, factor_before, equal_nan=True)
    ic = evaluate_rank_ic(factor, forward_returns, config, industry_labels=sample_industries, universe_mask=sample_usable)
    assert ic.rebalance_summary.valid_periods > 0
    direction = infer_long_direction(ic.rebalance_summary.mean)
    portfolio = long_portfolio_series(factor, forward_returns, ic.rebalance_indices, direction, config, industry_labels=sample_industries, universe_mask=sample_usable)
    performance = summarize_excess_returns(portfolio.excess_return, config)
    factor_results.append((expression, factor, ic, portfolio, performance))
elapsed = time.perf_counter() - start

for expression, _, ic, _, performance in factor_results:
    print(expression.to_formula())
    print('  IC:', ic.rebalance_summary)
    print('  Long:', performance)
print(f'两条表达式评价耗时: {elapsed:.3f}s')

In [ ]:
left = factor_results[0]
right = factor_results[1]
cross = factor_cross_sectional_correlation(left[1], right[1], min_count=20, industry_labels=sample_industries, universe_mask=sample_usable)
cross_summary = summarize_correlation(cross.values)
long_corr = excess_return_correlation(left[3].excess_return, right[3].excess_return)
assert cross_summary.valid_periods > 0
print('因子截面相关性:', cross_summary)
print('多头超额收益序列相关性:', long_corr)
print('阶段3验证通过：已处理数据能够完成表达式计算、缩尾、申万一级行业中性化、z-score、5日评价和相关性分析。')